# 🚀 Единый пайплайн разработки, актуализации и эксплуатации робота PRL/USDT (`v7.0`)

Полный 7-этапный цикл актуализации на свежих данных от синхронизации с VPS до боевого деплоя:
1. **Мониторинг VPS** (`5` коллекторов 24/7 + статус боевого демона)
2. **Синхронизация данных и разметка 6 горизонтов** (`H = 5:3, 12:3.5, 25:3, 60:3, 90:3, 120:3` + `mmap`-кэш)
3. **Дообучение скользящих окон роя** (`LightGBM` + 6 нейросетевых архитектур: `mlp_bottleneck`, `wavenet_tcn`, `cross_mixer`, `causal_cnn`, `attention`, `conv1d_gru` + сборка альянсов)
4. **Обновление 6-горизонтного RL-кэша и вспомогательных сетей `v7.0`** (`ExecutionRealismNet`, `CarouselNet v3` `47D`, `HazardNet v3` `47D`)
5. **Дообучение нативного 10-слотового RL-агента исполнения** (`MaskablePPO v7.0`, `49D`, динамические цели `+1/+2/+3 тика`)
6. **Валидация латентности (`< 350 мс`) и слепой OOS-экзамен**
7. **Безопасный деплой на боевой VPS (`81.19.137.18:2222`) с сохранением кошелька и перезапуском демона**

In [ ]:
# ЭТАП 1. Проверка статуса 5 коллекторов и боевого демона на VPS (81.19.137.18)
!.venv/bin/python -m _tools.monitor_vps --once
import _tools.live_rl_trader as lrt
lrt.show_live_status(tail_lines=20)

In [ ]:
# ЭТАП 2. Синхронизация свежих данных с VPS, пересборка Parquet, 6 горизонтов разметки (H=5..120s) и mmap-кэша
import importlib
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

v7p.step1_sync_and_rebuild_dataset(coin="prl")

In [ ]:
# ЭТАП 3. Дообучение новых скользящих окон роя (LightGBM + 6 нейросетей) по всем 6 горизонтам (H=5, 12, 25, 60, 90, 120s)
import importlib
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

v7p.step2_train_rolling_windows_6h(coin="prl", epochs=10)

In [ ]:
# ЭТАП 4. Пересборка 6-горизонтного RL-кэша (176ч) и дообучение ExecutionRealismNet, CarouselNet v3 (47D), HazardNet v3 (47D)
import importlib
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

v7p.step3_update_rl_cache_and_aux_nets(coin="prl", tail_hours=176.0, epochs=10)

In [ ]:
# ЭТАП 5. Дообучение нативного 10-слотового RL-агента исполнения MaskablePPO v7.0 (49D)
import importlib
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

res_ppo = v7p.step4_retrain_ppo_v70(coin="prl", tail_hours=176.0, ppo_epochs=12, steps_per_epoch=32768)

In [ ]:
# ЭТАП 6. Проверка латентности шага (< 350 мс) и слепой OOS-экзамен
import importlib
from IPython.display import display
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

df_ppo_trades, df_base_trades = v7p.step5_run_validation_gate(coin="prl", replay_steps=10)
display(df_ppo_trades.tail(15))

In [ ]:
# ЭТАП 7. Безопасный деплой обновлённых весов на боевой VPS (81.19.137.18:2222) с сохранением кошелька и перезапуском демона
import importlib
import _tools.v7_retrain_pipeline as v7p
importlib.reload(v7p)

v7p.step6_deploy_to_vps_and_verify(coin="prl")